# v3.0.0 Dementia AST Spectrograms — Primary Experiment

Primary Dementia screening experiment on Bridge2AI v3.0.0.
Adapted from v3 PD pipeline with key changes:
- Path: `features/torchaudio_mel_spectrogram.parquet` (column: `mel_spectrogram`)
- Mel bins: 60 (resized to 128 for AST)
- Hierarchical phenotype: `phenotype/diagnosis/cognitive_impairment.tsv` + `control.tsv`
- Cases: 74 dementia participants, Controls: 148 control participants (minus overlap)
- Task selection: 8 tasks with adequate Dementia + Control coverage
- Participant IDs: 6-digit numeric (zero-padded)

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

ROOT = Path('/data0/b2ai-voice/3.0.0')
SPEC = ROOT / 'features' / 'torchaudio_mel_spectrogram.parquet'
DEM_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'cognitive_impairment.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
DEMO_PATH = ROOT / 'phenotype' / 'demographics' / 'demographics.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Spec exists: {SPEC.exists()}')
print(f'Dementia phen exists: {DEM_PHEN.exists()}')
print(f'Ctrl phen exists: {CTRL_PHEN.exists()}')

In [ ]:
#2 - Load spectrogram data (row-group reads for memory efficiency)
pf = pq.ParquetFile(SPEC)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','mel_spectrogram','n_frames']).to_pandas())
spec = pd.concat(parts, ignore_index=True)

# Zero-pad participant IDs to 6 digits
spec['participant_id'] = spec['participant_id'].astype(str).str.zfill(6)

print(f'Total recordings: {len(spec)}')
print(f'Unique participants: {spec["participant_id"].nunique()}')
print(f'Unique tasks: {spec["task_name"].nunique()}')

In [ ]:
#3 - Build Dementia labels from hierarchical phenotype
dem_df = pd.read_csv(DEM_PHEN, sep='\t')
ctrl_df = pd.read_csv(CTRL_PHEN, sep='\t')

dem_ids = set(dem_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_df['participant_id'].astype(str).str.zfill(6))

# Remove any Dementia-Control overlap
overlap = dem_ids & ctrl_ids
ctrl_ids_clean = ctrl_ids - overlap
print(f'Dementia participants: {len(dem_ids)}')
print(f'Control participants (clean): {len(ctrl_ids_clean)}')
print(f'Overlap removed: {len(overlap)}')

# Assign labels
spec['label'] = np.nan
spec.loc[spec['participant_id'].isin(dem_ids), 'label'] = 1
spec.loc[spec['participant_id'].isin(ctrl_ids_clean), 'label'] = 0

# Keep only labeled recordings
data = spec.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
print(f'\nLabeled recordings: {len(data)}')
print(f'Dementia recordings: {(data["label"]==1).sum()}')
print(f'Control recordings: {(data["label"]==0).sum()}')
print(f'Unique participants: {data["participant_id"].nunique()}')

In [ ]:
#4 - Task selection: tasks with adequate Dementia + Control coverage
# Selected based on data exploration: tasks where both Dementia and Control
# participants have substantial representation
SELECTED_TASKS = [
    'prolonged-vowel',           # sustained phonation
    'glides-high-to-low',        # pitch control
    'glides-low-to-high',        # pitch control
    'diadochokinesis-pataka',    # articulatory agility
    'rainbow-passage',           # connected speech
    'picture-description',       # cognitive-linguistic
    'story-recall',              # memory + speech
    'maximum-phonation-time-1',  # sustained phonation
]

MIN_TIME_FRAMES = 100

data_selected = data[
    (data['task_name'].isin(SELECTED_TASKS)) &
    (data['n_frames'] >= MIN_TIME_FRAMES)
].copy()

print(f'Selected task recordings: {len(data_selected)}')
print(f'Dementia: {(data_selected["label"]==1).sum()} recordings from {data_selected[data_selected["label"]==1]["participant_id"].nunique()} participants')
print(f'Ctrl: {(data_selected["label"]==0).sum()} recordings from {data_selected[data_selected["label"]==0]["participant_id"].nunique()} participants')
print(f'Total participants: {data_selected["participant_id"].nunique()}')
print(f'\nPer-task counts:')
print(data_selected.groupby('task_name')['label'].value_counts().unstack(fill_value=0))

In [ ]:
#5 - Process spectrograms
from tqdm import tqdm

TARGET_SEQ_LEN = 1024

def process_spectrogram(spec_raw, target_len=1024):
    """Process raw spectrogram with reflect padding/center crop."""
    spec = np.stack(spec_raw).astype(np.float32)
    n_mels, time_len = spec.shape
    if time_len < target_len:
        pad_width = target_len - time_len
        spec = np.pad(spec, ((0, 0), (0, pad_width)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

X_list = []
for idx, row in tqdm(data_selected.iterrows(), total=len(data_selected), desc='Processing'):
    processed = process_spectrogram(row['mel_spectrogram'], TARGET_SEQ_LEN)
    X_list.append(processed)

X_raw = np.stack(X_list)
y_raw = data_selected['label'].values
participants_raw = data_selected['participant_id'].values
tasks_raw = data_selected['task_name'].values

print(f'Processed shape: {X_raw.shape}')  # (N, 60, 1024)
print(f'Value range: [{X_raw.min():.2f}, {X_raw.max():.2f}]')

In [ ]:
#6 - AST model definition and helpers
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig
from scipy.ndimage import zoom

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def resize_spectrogram(spec, target_mel=128, target_time=1024):
    """Resize spectrogram to AST expected dimensions (60 -> 128 mel bins)."""
    current_mel, current_time = spec.shape
    mel_ratio = target_mel / current_mel
    time_ratio = target_time / current_time
    resized = zoom(spec, (mel_ratio, time_ratio), order=1)
    return resized.astype(np.float32)

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(hidden_size=768, num_hidden_layers=12,
                             num_attention_heads=12, intermediate_size=3072,
                             max_length=1024, num_mel_bins=128)
            self.ast = ASTModel(config)
            hidden_size = 768
        if freeze_base:
            for param in self.ast.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 128, 1024) -> (B, 1024, 128)
        outputs = self.ast(input_values=x)
        pooled = outputs.pooler_output
        return self.classifier(pooled)

class ASTDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def evaluate_fold(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_parts = np.array(all_parts)
    unique_parts = np.unique(all_parts)
    part_probs, part_labels = [], []
    for p in unique_parts:
        mask = all_parts == p
        part_probs.append(all_probs[mask].mean())
        part_labels.append(all_labels[mask][0])
    return np.array(part_probs), np.array(part_labels), unique_parts

print('Model and helpers defined.')

In [ ]:
#7 - 5-Fold Cross-Validation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import copy
import time

unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])

print(f'Total participants: {len(unique_participants)}')
print(f'Dementia: {int(participant_labels.sum())} ({participant_labels.mean():.1%})')
print(f'Control: {int((participant_labels == 0).sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []
all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
all_oof_labels = participant_labels.astype(np.int64).copy()
norm_stats = {}

total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    X_train_fold = X_raw[train_mask]
    y_train_fold = y_raw[train_mask]
    parts_train_fold = participants_raw[train_mask]

    X_val_fold = X_raw[val_mask]
    y_val_fold = y_raw[val_mask]
    parts_val_fold = participants_raw[val_mask]

    print(f'Train: {len(X_train_fold)} recordings from {len(train_parts_fold)} participants')
    print(f'Val: {len(X_val_fold)} recordings from {len(val_parts_fold)} participants')

    # Resize 60 -> 128 mel bins
    X_train_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_train_fold, desc='resize train', leave=False)])
    X_val_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_val_fold, desc='resize val', leave=False)])

    # Fold-specific z-score normalization
    fold_mean = X_train_ast.mean()
    fold_std = X_train_ast.std()
    X_train_ast = (X_train_ast - fold_mean) / (fold_std + 1e-8)
    X_val_ast = (X_val_ast - fold_mean) / (fold_std + 1e-8)
    norm_stats[fold + 1] = {'mean': float(fold_mean), 'std': float(fold_std)}

    # Datasets
    train_ds = ASTDataset(X_train_ast, y_train_fold, parts_train_fold, augment=True)
    val_ds = ASTDataset(X_val_ast, y_val_fold, parts_val_fold, augment=False)

    # Balanced sampler
    class_counts = np.bincount(y_train_fold)
    weights = 1.0 / class_counts
    sample_weights = weights[y_train_fold]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    # Fresh model
    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)

    backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
    head_params = [p for n, p in model.named_parameters() if 'classifier' in n]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 5e-6, 'weight_decay': 0.01},
        {'params': head_params, 'lr': 5e-4, 'weight_decay': 0.01}
    ], betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

    # Dynamic class weights
    cc = np.bincount(y_train_fold)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    class_weights_fold = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=class_weights_fold, gamma=2.0)
    print(f'Class weights: {cw}')

    best_score = 0
    best_state = None
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(30):
        model.train()
        total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Evaluate
        part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
        if len(np.unique(part_labels_fold)) > 1:
            auc = roc_auc_score(part_labels_fold, part_probs)
            fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
            opt_idx = np.argmax(tpr - fpr)
            opt_thresh = thresholds[opt_idx]
            preds_opt = (part_probs >= opt_thresh).astype(int)
            f1_opt = f1_score(part_labels_fold, preds_opt, zero_division=0)
        else:
            auc, f1_opt = 0.5, 0.0

        score = 0.4 * auc + 0.6 * f1_opt
        if score > best_score + 0.01:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '<-- best'
        else:
            patience_counter += 1
            marker = ''

        avg_loss = total_loss / len(train_loader)
        print(f'  Epoch {epoch+1:02d} | loss {avg_loss:.4f} | AUC {auc:.3f} | F1opt {f1_opt:.3f} {marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best and get final predictions
    model.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)

    # Save fold model
    torch.save(model.state_dict(), str(RESULTS_DIR / f'ast_dementia_v3_fold{fold+1}.pt'))

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx_oof = np.where(unique_participants == pid)[0][0]
        all_oof_probs[idx_oof] = part_probs[i]

    # Fold metrics
    fold_auc = roc_auc_score(part_labels_fold, part_probs)
    fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
    opt_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[opt_idx]
    preds_opt = (part_probs >= opt_thresh).astype(int)

    fold_results.append({
        'fold': fold + 1,
        'auc': float(fold_auc),
        'f1_opt': float(f1_score(part_labels_fold, preds_opt, zero_division=0)),
        'recall_opt': float(recall_score(part_labels_fold, preds_opt, zero_division=0)),
        'precision_opt': float(precision_score(part_labels_fold, preds_opt, zero_division=0)),
        'threshold': float(opt_thresh)
    })

    print(f'Fold {fold+1}: AUC={fold_auc:.4f}, F1={fold_results[-1]["f1_opt"]:.4f}')

    del model, optimizer, train_ds, val_ds
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f'\nTotal CV time: {total_time/60:.1f} minutes')

In [ ]:
#8 - CV Summary and OOF evaluation
from scipy import stats
from sklearn.metrics import confusion_matrix, accuracy_score

aucs = [r['auc'] for r in fold_results]
f1s = [r['f1_opt'] for r in fold_results]
recalls = [r['recall_opt'] for r in fold_results]
precisions = [r['precision_opt'] for r in fold_results]

n = N_FOLDS
t_crit = stats.t.ppf(0.975, df=n - 1)

print('Per-fold results:')
for r in fold_results:
    print(f'  Fold {r["fold"]}: AUC={r["auc"]:.4f}  F1={r["f1_opt"]:.4f}  '
          f'Rec={r["recall_opt"]:.4f}  Prec={r["precision_opt"]:.4f}  Thr={r["threshold"]:.3f}')

print('\nMean +/- SD [95% CI]:')
for name, arr in [('AUC', aucs), ('F1', f1s), ('Recall', recalls), ('Precision', precisions)]:
    m = np.mean(arr)
    sd = np.std(arr, ddof=1)
    se = sd / np.sqrt(n)
    ci_lo = m - t_crit * se
    ci_hi = m + t_crit * se
    print(f'  {name}: {m:.4f} ({sd:.4f}) [{ci_lo:.4f}, {ci_hi:.4f}]')

# OOF metrics
oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr, tpr, thresholds = roc_curve(all_oof_labels, all_oof_probs)
opt_idx = np.argmax(tpr - fpr)
oof_thresh = thresholds[opt_idx]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)

oof_f1 = f1_score(all_oof_labels, oof_preds, zero_division=0)
oof_rec = recall_score(all_oof_labels, oof_preds, zero_division=0)
oof_prec = precision_score(all_oof_labels, oof_preds, zero_division=0)
oof_acc = accuracy_score(all_oof_labels, oof_preds)
cm = confusion_matrix(all_oof_labels, oof_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print(f'\nOOF AUC:         {oof_auc:.4f}')
print(f'OOF F1:          {oof_f1:.4f}')
print(f'OOF Recall:      {oof_rec:.4f}')
print(f'OOF Precision:   {oof_prec:.4f}')
print(f'OOF Specificity: {specificity:.4f}')
print(f'OOF Accuracy:    {oof_acc:.4f}')
print(f'Threshold:       {oof_thresh:.3f}')
print(f'\nConfusion Matrix: TP={tp} FP={fp} FN={fn} TN={tn}')

# Save results
np.savez(str(RESULTS_DIR / 'ast_dementia_v3_cv_results.npz'),
    oof_probs=all_oof_probs,
    oof_labels=all_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array(aucs),
    fold_f1s=np.array(f1s),
    fold_recalls=np.array(recalls),
    fold_precisions=np.array(precisions),
    fold_thresholds=np.array([r['threshold'] for r in fold_results]),
    oof_auc=np.array(oof_auc),
    oof_thresh=np.array(oof_thresh),
    norm_means=np.array([norm_stats[f+1]['mean'] for f in range(N_FOLDS)]),
    norm_stds=np.array([norm_stats[f+1]['std'] for f in range(N_FOLDS)]),
)
print(f'\nSaved: {RESULTS_DIR}/ast_dementia_v3_cv_results.npz')
print(f'Saved: {RESULTS_DIR}/ast_dementia_v3_fold{{1-5}}.pt')

In [ ]:
#9 - Metadata evaluation (age + sex logistic regression + late fusion)
from sklearn.linear_model import LogisticRegression

demo = pd.read_csv(DEMO_PATH, sep='\t')
demo['participant_id'] = demo['participant_id'].astype(str).str.zfill(6)

# Build metadata features for our cohort
meta_df = pd.DataFrame({'participant_id': unique_participants, 'label': participant_labels})
meta_df = meta_df.merge(demo[['participant_id', 'age', 'sex_at_birth']], on='participant_id', how='left')
meta_df['sex_numeric'] = (meta_df['sex_at_birth'] == 'Male').astype(int)
meta_df = meta_df.dropna(subset=['age', 'sex_numeric'])

print(f'Participants with metadata: {len(meta_df)}')

# 5-fold CV for metadata-only model (same splits)
meta_participants = meta_df['participant_id'].values
meta_labels = meta_df['label'].values
meta_features = meta_df[['age', 'sex_numeric']].values

meta_oof_probs = np.zeros(len(meta_df), dtype=np.float32)

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf_meta.split(meta_participants, meta_labels)):
    lr = LogisticRegression(class_weight='balanced', max_iter=1000)
    lr.fit(meta_features[train_idx], meta_labels[train_idx])
    meta_oof_probs[val_idx] = lr.predict_proba(meta_features[val_idx])[:, 1]

meta_auc = roc_auc_score(meta_labels, meta_oof_probs)
fpr_m, tpr_m, thr_m = roc_curve(meta_labels, meta_oof_probs)
opt_idx_m = np.argmax(tpr_m - fpr_m)
meta_preds = (meta_oof_probs >= thr_m[opt_idx_m]).astype(int)
meta_f1 = f1_score(meta_labels, meta_preds, zero_division=0)

print(f'\nMetadata-only AUC: {meta_auc:.4f}')
print(f'Metadata-only F1:  {meta_f1:.4f}')

# Late fusion: combine AST probs + metadata probs
# Match participants between AST and metadata
ast_prob_map = dict(zip(unique_participants, all_oof_probs))
fused_probs = []
fused_labels = []
for i, pid in enumerate(meta_participants):
    if pid in ast_prob_map:
        fused_probs.append(0.5 * ast_prob_map[pid] + 0.5 * meta_oof_probs[i])
        fused_labels.append(meta_labels[i])

fused_probs = np.array(fused_probs)
fused_labels = np.array(fused_labels)
fused_auc = roc_auc_score(fused_labels, fused_probs)
fpr_f, tpr_f, thr_f = roc_curve(fused_labels, fused_probs)
opt_idx_f = np.argmax(tpr_f - fpr_f)
fused_preds = (fused_probs >= thr_f[opt_idx_f]).astype(int)
fused_f1 = f1_score(fused_labels, fused_preds, zero_division=0)

print(f'\nAST+Metadata AUC:  {fused_auc:.4f}')
print(f'AST+Metadata F1:   {fused_f1:.4f}')

print(f'\nSummary:')
print(f'  AST only:     AUC={oof_auc:.4f}  F1={oof_f1:.4f}')
print(f'  Metadata only: AUC={meta_auc:.4f}  F1={meta_f1:.4f}')
print(f'  AST+Metadata: AUC={fused_auc:.4f}  F1={fused_f1:.4f}')

In [ ]:
#10 - Per-task contribution analysis
# Re-run inference on each task separately using fold models
import json

task_results = {}

for task in SELECTED_TASKS:
    task_mask = tasks_raw == task
    if task_mask.sum() == 0:
        continue
    X_task = X_raw[task_mask]
    y_task = y_raw[task_mask]
    p_task = participants_raw[task_mask]

    # Recording-level AUC from OOF: match each recording's participant to their OOF prob
    rec_probs = []
    rec_labels = []
    for i, pid in enumerate(p_task):
        idx_p = np.where(unique_participants == pid)[0]
        if len(idx_p) > 0:
            rec_probs.append(all_oof_probs[idx_p[0]])
            rec_labels.append(y_task[i])

    if len(np.unique(rec_labels)) > 1:
        task_auc = roc_auc_score(rec_labels, rec_probs)
    else:
        task_auc = float('nan')

    task_results[task] = {
        'recordings': int(task_mask.sum()),
        'participants': len(np.unique(p_task)),
        'auc': float(task_auc)
    }
    print(f'{task}: {task_mask.sum()} recs, {len(np.unique(p_task))} ppl, AUC={task_auc:.4f}')

# Save task results
with open(str(RESULTS_DIR / 'ast_dementia_v3_per_task.json'), 'w') as f:
    json.dump(task_results, f, indent=2)
print(f'\nSaved: {RESULTS_DIR}/ast_dementia_v3_per_task.json')